#### Ejercicio 01

¿Qué tipo de cómputo aparece disponible (serverless, all-purpose, job cluster)?

R/ En Databricks Free Edition aparece disponible únicamente un cómputo serverless de tipo All‑Purpose, el cual es administrado completamente por Databricks.

¿Puede modificar la configuración del cluster? 
R/ No, no es posible modificar la configuración del cluster ni crear nuevos clusters manualmente en Free Edition.
¿Por qué cree que es así en Free
Edition?
R/ Esto se debe a que Free Edition está orientada al aprendizaje. Databricks simplifica la infraestructura para evitar configuraciones complejas y permitir que el usuario se enfoque en aprender a usar la plataforma y Spark.
¿Qué ventaja tiene el cómputo serverless para un usuario que está aprendiendo?
R/ La principal ventaja es que no se necesita conocimiento de administración de infraestructura. El usuario puede ejecutar notebooks y código directamente sin preocuparse por el manejo del cluster.

In [0]:
# Información del entorno de cómputo
print("Versión de Spark:", spark.version)

# Configuraciones del entorno
configs = [
    "spark.databricks.clusterUsageTags.clusterName",
    "spark.databricks.clusterUsageTags.clusterNodeType",
    "spark.databricks.clusterUsageTags.sparkVersion",
    "spark.databricks.clusterUsageTags.clusterAllTags",
]
for c in configs:
    try:
        val = spark.conf.get(c)
        print(f"{c}: {val}")
    except:
        print(f"{c}: no disponible")

# Catálogo y schema por defecto
print("Catálogo actual:", spark.catalog.currentCatalog())
print("Schema actual:", spark.catalog.currentDatabase())

#### Ejercicio 02
¿Qué SQL Warehouse aparece disponible? ¿Cuál es su tamaño (size)?
R/ En Databricks Free Edition aparece disponible un SQL Warehouse serverless de tamaño Small.

¿Cuál es la diferencia entre un All-Purpose Compute y un SQL Warehouse? ¿Cuándo usaría cada uno?
R/ All-Purpose Compute está diseñado para ejecutar notebooks, trabajos interactivos y procesamiento de datos con Spark. SQL Warehouse está optimizado para ejecutar consultas SQL y análisis de datos estructurados. Usaría All-Purpose Compute para tareas de ingeniería de datos y machine learning, y SQL Warehouse para análisis y visualización de datos con SQL.

¿Qué significa que el warehouse sea "serverless"?
R/ Significa que la infraestructura es administrada automáticamente por Databricks, sin necesidad de configurar o gestionar recursos manualmente. El usuario puede ejecutar consultas SQL sin preocuparse por la administración del hardware.

#### Ejercicio 03

¿Cuál es la ruta completa del archivo que creó?
R/ La ruta completa del archivo creado en un Volume de Unity Catalog es: `/Volumes/<catalog>/<schema>/<volume>/<archivo>`. Por ejemplo: `/Volumes/main/default/my_volume/mi_archivo.csv`.

¿En qué se diferencia un Volume de Unity Catalog del antiguo DBFS? (Pista: piense en gobernanza, permisos y namespace de 3 niveles)
R/ Un Volume de Unity Catalog ofrece gobernanza centralizada, control granular de permisos y un namespace de 3 niveles (catalogo, esquema, volumen), mientras que DBFS no tiene estas capacidades avanzadas de seguridad y organización. Unity Catalog permite definir quién puede acceder y modificar datos, facilitando el cumplimiento y la administración de datos.


In [0]:
# Crear un schema para el lab (si no existe)
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.default")
spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA default")

# Crear un Volume para almacenar archivos
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.lab5_archivos")

# Verificar que el Volume existe
display(spark.sql("SHOW VOLUMES IN workspace.default"))



# Escribir un archivo de prueba en el Volume
volume_path = "/Volumes/workspace/default/lab5_archivos"

dbutils.fs.put(
    f"{volume_path}/prueba.txt",
    "Hola desde el Lab 5 - Arquitectura de Databricks\n"
    "Este archivo está almacenado en un Unity Catalog Volume.",



    overwrite=True
)

# Leer el archivo
contenido = dbutils.fs.head(f"{volume_path}/prueba.txt")
print(contenido)

# Listar archivos en el Volume
display(dbutils.fs.ls(volume_path))

#### Ejercicio 04

In [0]:
from pyspark.sql.types import *

# Datos sintéticos: tiendas
tiendas_data = [
    (1, "Tienda Central", "San José", "CR"),
    (2, "Tienda Norte", "Heredia", "CR"),
    (3, "Tienda Sur", "Cartago", "CR"),
    (4, "Tienda Playa", "Guanacaste", "CR"),
    (5, "Tienda Digital", "Online", "CR"),
]

tiendas_schema = StructType([
    StructField("tienda_id", IntegerType(), False),
    StructField("nombre", StringType(), False),
    StructField("ubicacion", StringType(), False),
    StructField("pais", StringType(), False),
])

df_tiendas = spark.createDataFrame(tiendas_data, tiendas_schema)
display(df_tiendas)

In [0]:
from pyspark.sql.functions import *
from datetime import date

# Datos sintéticos: ventas (100 registros)
import random
random.seed(42)

In [0]:
import builtins

ventas_rows = []
productos = ["Laptop", "Mouse", "Teclado", "Monitor", "Audífonos",
             "USB Hub", "Webcam", "Cargador", "Cable HDMI", "Mochila"]
for i in range(1, 101):
    ventas_rows.append((
        i,
        random.choice([1, 2, 3, 4, 5]),
        random.choice(productos),
        random.randint(1, 5),
        builtins.round(random.uniform(10.0, 500.0), 2),
        date(2026, random.randint(1, 3), random.randint(1, 28)).isoformat(),
    ))

ventas_schema = StructType([
    StructField("venta_id", IntegerType(), False),
    StructField("tienda_id", IntegerType(), False),
    StructField("producto", StringType(), False),
    StructField("cantidad", IntegerType(), False),
    StructField("precio_unitario", DoubleType(), False),
    StructField("fecha", StringType(), False),
])

df_ventas = spark.createDataFrame(ventas_rows, ventas_schema)
display(df_ventas)

#### Ejercicio 05

In [0]:
# Crear un schema dentro del catalog workspace
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.lab5_tienda")
spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA lab5_tienda")

In [0]:
# Guardar tiendas como tabla Delta
df_tiendas.write.mode("overwrite").saveAsTable("workspace.lab5_tienda.tiendas")

# Guardar ventas como tabla Delta
df_ventas.write.mode("overwrite").saveAsTable("workspace.lab5_tienda.ventas")

print("Tablas creadas exitosamente")

# Listar tablas en el schema
display(spark.sql("SHOW TABLES IN workspace.lab5_tienda"))

#### Ejercicio 06

¿Cuál es el nombre completo (3 niveles) de la tabla ventas ? (formato: catalog.schema.table )
R/ El nombre completo es: `main.default.ventas`, donde `main` es el catálogo, `default` es el esquema y `ventas` es el nombre de la tabla.

¿Qué formato de almacenamiento usa la tabla? (Busque en la salida de DESCRIBE EXTENDED)
R/ La tabla usa el formato `Delta`, que permite realizar actualizaciones, versionado y manejo eficiente de datos.

¿Por qué esta estructura de 3 niveles es útil en una organización grande?
R/ Esta estructura ayuda a organizar los datos de manera clara: el catálogo agrupa áreas o proyectos, el esquema separa conjuntos de datos dentro del catálogo, y la tabla contiene los datos. Así, se facilita el control de acceso, la gobernanza y la administración de grandes volúmenes de información.


In [0]:
# Nivel 1: listar catalogs disponibles
display(spark.sql("SHOW CATALOGS"))

# Nivel 2: listar schemas en el catalog actual
display(spark.sql("SHOW SCHEMAS IN workspace"))

# Nivel 3: listar tablas en el schema lab5_tienda
display(spark.sql("SHOW TABLES IN workspace.lab5_tienda"))

# Describir una tabla con detalle
display(spark.sql("DESCRIBE EXTENDED workspace.lab5_tienda.ventas"))

#### Ejrcicio 07
¿Cuál es la tienda con mayor ingreso total?
R/ Tienda Sur Cartago

¿Para qué tipo de usuario empresarial sería útil esta vista Gold?
R/ Esta vista Gold es útil para analistas de negocio, gerentes de ventas y directores, ya que les permite ver fácilmente el desempeño de cada tienda y tomar decisiones basadas en datos.

¿Cuál es la ventaja de usar una vista en lugar de crear otra tabla?
R/ Usar una vista permite acceder a datos actualizados sin duplicar información. Las vistas se actualizan automáticamente cuando cambian los datos originales, lo que ahorra espacio y facilita el mantenimiento.


In [0]:
# Vista Gold: ventas por tienda con métricas
spark.sql('''
    CREATE OR REPLACE VIEW workspace.lab5_tienda.ventas_por_tienda_gold AS
    SELECT
        t.nombre AS tienda,
        t.ubicacion,
        COUNT(v.venta_id) AS total_transacciones,
        SUM(v.cantidad) AS unidades_vendidas,
        ROUND(SUM(v.cantidad * v.precio_unitario), 2) AS ingreso_total,
        ROUND(AVG(v.precio_unitario), 2) AS precio_promedio
    FROM workspace.lab5_tienda.ventas v
    JOIN workspace.lab5_tienda.tiendas t ON v.tienda_id = t.tienda_id
    GROUP BY t.nombre, t.ubicacion
    ORDER BY ingreso_total DESC
''')

display(spark.sql("SELECT * FROM workspace.lab5_tienda.ventas_por_tienda_gold"))

#### Ejercicio 08

¿Cuántas versiones tiene la tabla ventas según DESCRIBE HISTORY?
R/ La tabla ventas tiene tantas versiones como veces se ha modificado (insertado, actualizado o borrado datos). Puedes ver el número de versiones ejecutando `DESCRIBE HISTORY main.default.ventas`, donde cada fila representa una versión diferente.

¿Qué diferencia hay entre una TABLE y una VIEW en la salida de information_schema?
R/ Una TABLE es un objeto que almacena datos físicamente, mientras que una VIEW es una consulta guardada que muestra datos de otras tablas, pero no almacena datos propios. En information_schema, las TABLES aparecen en la tabla `tables` y las VIEWS en la tabla `views`.

¿De qué sirve el information_schema para un Data Engineer?
R/ El information_schema permite consultar metadatos sobre las tablas, vistas y otros objetos de la base de datos. Es útil para entender la estructura, permisos y relaciones entre los datos, facilitando la administración y el análisis de la información.


In [0]:
# Historial de cambios de la tabla ventas
display(spark.sql("DESCRIBE HISTORY workspace.lab5_tienda.ventas"))

# Detalle de la tabla (formato, ubicación, propiedades)
display(spark.sql("DESCRIBE DETAIL workspace.lab5_tienda.ventas"))

# Propiedades extendidas de la tabla tiendas
display(spark.sql("DESCRIBE EXTENDED workspace.lab5_tienda.tiendas"))

# Listar todo: tablas + vistas en el schema
display(spark.sql("SHOW TABLES IN workspace.lab5_tienda"))

# Ver las vistas específicamente
display(spark.sql(
    "SELECT table_name, table_type "
    "FROM workspace.information_schema.tables "
     "WHERE table_schema = 'lab5_tienda'"
))

#### Ejercicio 09

¿Qué objetos ve dentro de lab5_tienda en el Catalog Explorer? (tablas, vistas, volumes)
R/ Dentro de lab5_tienda en el Catalog Explorer se pueden diferentes tipos de objetos:
- Tablas: donde se almacenan los datos.
- Vistas: consultas guardadas que muestran datos de otras tablas.
- Volumes: espacios de almacenamiento para archivos y datos externos.

¿Qué tablas alimentan la vista ventas_por_tienda_gold?
R/ La vista ventas_por_tienda_gold se alimenta de las tablas que contienen información de ventas y tiendas, como la tabla ventas y la tabla tiendas. Estas tablas dan los datos que la vista utiliza para mostrar el ingreso total por tienda.

¿Cómo ayuda el Catalog Explorer y el lineage a entender el flujo de datos en un proyecto real?
R/ El Catalog Explorer te permite ver todos los objetos y cómo están organizados. El lineage muestra cómo los datos se mueven y transforman desde las tablas originales hasta las vistas finales. Esto ayuda a entender de dónde vienen los datos, cómo se procesan y cómo llegan a los reportes, facilitando el análisis y la auditoría en proyectos reales.
